# Info on original Dataset
get patients from it, how many are only in label and only in omics or clinical. get wsi info, how many, how many of those are in the dataset, if in dataset are patients without wsi or withour omics data

## get original dataset

In [41]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from random import shuffle
from functools import reduce
import copy
from collections import Counter

In [42]:
# get the whole data
study = 'UCEC'
signatures = 'other'

if signatures == 'other':
    omics_data_c = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/other/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )
    omics_data_h = omics_data_c
    omics_data_x = omics_data_c

else:
    # omics data
    omics_data_c = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/combine/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

    omics_data_h = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/hallmarks/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

    omics_data_x = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/xena/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

# metadata
label_data = pd.read_csv(f'src/datasets_csv/metadata/tcga_{study}.csv', low_memory=False)    # contains pairing between case_id and slide_id(s) + info on survival

# clinical data
clinical_data = pd.read_csv(f'src/datasets_csv/clinical_data/tcga_{study}_clinical.csv', index_col=0) # clinical data


label data

In [43]:
# get all patients
all_pat_label = label_data['case_id'].tolist()
print(f'Total number of patients in label dataset: {len(all_pat_label)}')
all_pat_label_unique = list(set(all_pat_label))
print(f'Total number of unique patients in label dataset: {len(all_pat_label_unique)}')

single_wsi_pat_label = [i for i in set(all_pat_label) if all_pat_label.count(i) == 1]
print(f'Number of patients with only one WSI: {len(single_wsi_pat_label)}')
more_wsi_pat_label = [i for i in set(all_pat_label) if all_pat_label.count(i) >1]
print(f'Number of patients with more than one WSI: {len(more_wsi_pat_label)}')
double_wsi_pat_label = [i for i in set(all_pat_label) if all_pat_label.count(i) == 2]
print(f'Number of patients with two WSI: {len(double_wsi_pat_label)}')
triple_wsi_pat_label = [i for i in set(all_pat_label) if all_pat_label.count(i) == 3]
print(f'Number of patients with three WSI: {len(triple_wsi_pat_label)}')
quadruple_wsi_pat_label = [i for i in set(all_pat_label) if all_pat_label.count(i) == 4]
print(f'Number of patients with four WSI: {len(quadruple_wsi_pat_label)}')
quintuple_wsi_pat_label = [i for i in set(all_pat_label) if all_pat_label.count(i) == 5]
print(f'Number of patients with five WSI: {len(quintuple_wsi_pat_label)}')

# check the quantities:
'''
print('Checking the quantities:')
tot = (len(single_wsi_pat_label) * 1 + len(double_wsi_pat_label) * 2 + len(triple_wsi_pat_label) * 3 +
       len(quadruple_wsi_pat_label) * 4 + len(quintuple_wsi_pat_label) * 5)
print(f'Total number of patients not unique in label dataset: {tot}')
tot2 = (len(single_wsi_pat_label) + len(double_wsi_pat_label) + len(triple_wsi_pat_label) +
       len(quadruple_wsi_pat_label) + len(quintuple_wsi_pat_label))
print(f'Total number of unique patients in label dataset: {tot2}')
'''

# chcek that patients with multple rows actually have different slide ids
for pat in more_wsi_pat_label:
       slide_ids = label_data[label_data['case_id'] == pat]['slide_id'].tolist()
       if len(slide_ids) != len(set(slide_ids)):
              print(f'Patient {pat} has multiple rows with the same slide id!')

print('all patients with multiple WSIs have all different slide ids')


Total number of patients in label dataset: 486
Total number of unique patients in label dataset: 486
Number of patients with only one WSI: 486
Number of patients with more than one WSI: 0
Number of patients with two WSI: 0
Number of patients with three WSI: 0
Number of patients with four WSI: 0
Number of patients with five WSI: 0
all patients with multiple WSIs have all different slide ids


omics data

In [ ]:
print('--- OMICS ---')
all_pat_omics = omics_data_x.index.tolist()
print(f'Total number of patients in omics dataset: {len(all_pat_omics)}')
all_pat_omics_unique = list(set(all_pat_omics))
print(f'Total number of unique patients in omics dataset: {len(all_pat_omics_unique)}')

single_wsi_pat_omics = [i for i in set(all_pat_omics) if all_pat_omics.count(i) == 1]
print(f'Number of patients with only one OMIC: {len(single_wsi_pat_omics)}')
more_wsi_pat_omics = [i for i in set(all_pat_omics) if all_pat_omics.count(i) >1]
print(f'Number of patients with more than one OMIC: {len(more_wsi_pat_omics)}')
double_wsi_pat_omics = [i for i in set(all_pat_omics) if all_pat_omics.count(i) == 2]
print(f'Number of patients with two OMIC: {len(double_wsi_pat_omics)}')
print(f'patients with two OMIC: {double_wsi_pat_omics}')
if len(double_wsi_pat_omics) > 0: 
    print(omics_data_x.loc[double_wsi_pat_omics[0]])



--- OMICS ---
Total number of patients in omics dataset: 487
Total number of unique patients in omics dataset: 487
Number of patients with only one OMIC: 487
Number of patients with more than one OMIC: 0
Number of patients with two OMIC: 0
patients with two OMIC: []
-- info more specific --
OMIC type: miRNA
Number of attributes for miRNA: 203
Number of patients with values for miRNA attributes: 487
OMIC type: DNA
Number of attributes for DNA: 3176
Number of patients with values for DNA attributes: 487
OMIC type: mRNA
Number of attributes for mRNA: 4096
Number of patients with values for mRNA attributes: 487
OMIC type: CNV
Number of attributes for CNV: 4096
Number of patients with values for CNV attributes: 381


clinicla data

In [45]:
all_pat_clinical = clinical_data['case_id'].tolist()
print(f'Total number of patients in clinical dataset: {len(all_pat_clinical)}')
all_pat_clinical_unique = list(set(all_pat_clinical))
print(f'Total number of unique patients in clinical dataset: {len(all_pat_clinical_unique)}')

single_wsi_pat_clinical = [i for i in set(all_pat_clinical) if all_pat_clinical.count(i) == 1]
print(f'Number of patients with only one CLINICAL: {len(single_wsi_pat_clinical)}')
more_wsi_pat_clinical = [i for i in set(all_pat_clinical) if all_pat_clinical.count(i) >1]
print(f'Number of patients with more than one CLINICAL: {len(more_wsi_pat_clinical)}')
double_wsi_pat_clinical = [i for i in set(all_pat_clinical) if all_pat_clinical.count(i) == 2]
print(f'Number of patients with two CLINICAL: {len(double_wsi_pat_clinical)}')

Total number of patients in clinical dataset: 487
Total number of unique patients in clinical dataset: 487
Number of patients with only one CLINICAL: 487
Number of patients with more than one CLINICAL: 0
Number of patients with two CLINICAL: 0


see how many patients are present only in one of the datasets

In [ ]:
# - removes every element from the first set that also exists in the second set, giving only the unique elements from the first list.
label_not_omics = list(set(all_pat_label_unique) - set(all_pat_omics_unique))
print(f'Number patients in label but not in omics: {len(label_not_omics)}')
if len(label_not_omics) > 0:
    print(f'missing patients: {label_not_omics}\n')


omics_not_label = list(set(all_pat_omics_unique) - set(all_pat_label_unique))
print(f'Number patients in omics but not in label: {len(omics_not_label)}')
if len(omics_not_label) > 0:
    print(f'missing patients: {omics_not_label}\n')


# patients present in both omics and label
common_patients = list(set(all_pat_label_unique).intersection(set(all_pat_omics_unique)))
print(f'Number of patients present in both omics and label datasets: {len(common_patients)}')

Number patients in label but not in omics: 0
Number patients in omics but not in label: 1
missing patients: ['TCGA-AJ-A3BH']


more specific
Number patients in label but not in omics with DNA values: 0
Number patients in label but not in omics with CNV values: 106
Number patients in label but not in omics with miRNA values: 0
Number patients in label but not in omics with mRNA values: 0
Number patients in label but not DNA or CNV: 0
Number of patients present in both omics and label datasets: 486


In [47]:
# get patients present in label but not in clinical data
label_not_clinical = list(set(all_pat_label_unique) - set(all_pat_clinical_unique))
print(f'Number patients in label but not in clinical data: {len(label_not_clinical)}')
if len(label_not_clinical) > 0:
    print(f'missing patients: {label_not_clinical}\n')

# patients present in clinical but not in label data
clinical_not_label = list(set(all_pat_clinical_unique) - set(all_pat_label_unique))
print(f'Number patients in clinical but not in label data: {len(clinical_not_label)}')
if len(clinical_not_label) > 0:
    print(f'missing patients: {clinical_not_label}\n')

# patients present in both clinical and label
common_patients_clinical = list(set(all_pat_label_unique).intersection(set(all_pat_clinical_unique)))
print(f'Number of patients present in both clinical and label datasets: {len(common_patients_clinical)}')

Number patients in label but not in clinical data: 0
Number patients in clinical but not in label data: 1
missing patients: ['TCGA-AJ-A3BH']

Number of patients present in both clinical and label datasets: 486


## get wsi data

### utils

In [48]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split

from src.zoorvival.data import load_tcga_data
from src.zoorvival.nn.training import as_torch_dataset

%reload_ext autoreload
%autoreload 2

np.random.seed(42)

In [49]:
PROJECT = study
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [50]:
def request_tcga_file_info(data_type: str, project_ids=None) -> pd.DataFrame:
    """
    Get TCGA file information from GDC API.

    Args:
        data_type (str): The type of data to retrieve.
        project_ids (list[str], optional): List of project IDs to filter by. Defaults to None.
    
    Returns:
        pd.DataFrame: DataFrame containing file information.
    """
    if project_ids is None:
        project_ids = get_tcga_projects()
    fields = [
        "file_name",
        "md5sum",
        "cases.submitter_id",
        "cases.samples.sample_type",
        "cases.project.project_id",
        "cases.project.primary_site",
    ]
    fields = ",".join(fields)
    files_endpt = "https://api.gdc.cancer.gov/files"
    filters = {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {
                    "field": "files.experimental_strategy",
                    "value": [data_type],
                },
            },
            {
                "op": "in",
                "content": {
                    "field": "cases.project.project_id",
                    "value": project_ids,
                },
            },
        ],
    }
    params = {"filters": filters, "fields": fields, "format": "TSV", "size": "200000"}
    response = requests.post(
        files_endpt, headers={"Content-Type": "application/json"}, json=params
    )
    return pd.read_csv(StringIO(response.content.decode("utf-8")), sep="\t")


def process_tcga_file_info(df_files, suffix, data_type) -> pd.DataFrame:
    rename_cols = {
        "cases.0.project.project_id": "project_id",
        "cases.0.project.primary_site": "primary_site",
        "cases.0.submitter_id": "submitter_id",
        "cases.0.samples.0.sample_type": "sample_type",
        "file_name": f"{data_type}_file_name",
        "id": f"{data_type}_file_id",
        "md5sum": f"{data_type}_md5sum",
    }
    df_files = df_files.rename(columns=rename_cols)
    df_files = df_files[df_files[f"{data_type}_file_name"].str.endswith(suffix)]
    df_files = df_files[df_files["sample_type"] == "Primary Tumor"]
    df_files = df_files[~df_files.duplicated(subset=["submitter_id"], keep="first")]
    return df_files


def get_suffix_counts(df_files) -> pd.Series:
    file_col = [c for c in df_files.columns if "file_name" in c][0]
    file_suffixes = [".".join(f.split(".")[-2:]) for f in df_files[file_col]]
    suffix_counts = pd.Series(file_suffixes).value_counts()
    suffix_counts = suffix_counts[suffix_counts > 1]
    suffix_counts = suffix_counts.reset_index()
    return suffix_counts

def get_tcga_projects() -> list[str]:
    """
    Get a list of TCGA projects from the GDC API.
    """
    response = requests.get("https://api.gdc.cancer.gov/projects", params={"size": 10000})
    response.raise_for_status()
    projects = response.json()["data"]["hits"]
    projects = [proj for proj in projects if proj.get("project_id", "").startswith("TCGA-")]
    projects = [proj["id"] for proj in projects]
    return projects

## comparison

In [51]:
db = load_tcga_data(PROJECT)

<KeysViewHDF5 ['embeddings', 'medoid_embeddings', 'medoid_indices', 'pooled_labels', 'tiles_info']>
[[ 6544. 46208.]
 [ 6544. 47232.]
 [ 6544. 48256.]
 ...
 [99728. 49280.]
 [99728. 50304.]
 [99728. 51328.]]


In [52]:
train_patients = set('TCGA-' + i for i in db.train.df_clinical.index.astype(str))
test_patients = set('TCGA-' + i for i in db.test.df_clinical.index.astype(str))

wsi_patients = train_patients.union(test_patients)
print('number of wsi present in wsi database: ', len(wsi_patients))

number of wsi present in wsi database:  486


In [53]:
# - removes every element from the first set that also exists in the second set, giving only the unique elements from the first list.
label_not_wsi = list(set(all_pat_label_unique) - set(wsi_patients))
print(f'Number patients in label but not in wsi: {len(label_not_wsi)}')
if len(label_not_wsi) > 0:
    print(f'missing patients: {label_not_wsi}\n')


wsi_not_label = list(set(wsi_patients) - set(all_pat_label_unique))
print(f'Number patients in wsi but not in label: {len(wsi_not_label)}')
if len(wsi_not_label) > 0:
    print(f'missing patients: {wsi_not_label}\n')


# patients present in both wsi and label
common_patients = list(set(all_pat_label_unique).intersection(set(wsi_patients)))
print(f'Number of patients present in both wsi and label datasets: {len(common_patients)}')

Number patients in label but not in wsi: 0
Number patients in wsi but not in label: 0
Number of patients present in both wsi and label datasets: 486


there is the problem of only having one wsi per patient in wsi db: in wsi db the wsi are saved with the case_id of the patient.
Need to see if needed to remove some rows from label data, or if we can just collect only one wsi from the db: need to check if the survival info on label is duplicated, or different

In [54]:
print(more_wsi_pat_label)
for pat in more_wsi_pat_label:
    rows = label_data[label_data['case_id'] == pat]
    rows = rows.drop(columns=['slide_id', 'Unnamed: 0'])
    #print(rows)
    # finding the duplicates and removing them
    doubles = rows.duplicated(keep=False)
    rows = rows[~doubles]
    #print('patient ', pat, rows.shape)
    if rows.shape[0] > 1:
        print('a patient has different survivval data for different slides')
    else:
        print('patient ', pat, ' ok')

[]


the rows seem to be identical: we can keep the labels --> find a way to not return 2 times the same wsi, but we can keep the patients. Not all instances of label_data contain patients that will be used during the training

patients in label but with no omics or wsi:

In [55]:
'''
# - removes every element from the first set that also exists in the second set, giving only the unique elements from the first list.
label_not_wsi_not_omics = list(set(label_not_wsi) - set(label_not_omics))
print(f'Number patients in label not in wsi: {len(label_not_wsi_not_omics)}')
if len(label_not_wsi_not_omics) > 0:
    print(f'missing patients: {label_not_wsi_not_omics}\n')


wsi_not_label = list(set(wsi_patients) - set(all_pat_label_unique))
print(f'Number patients in wsi but not in label: {len(wsi_not_label)}')
if len(wsi_not_label) > 0:
    print(f'missing patients: {wsi_not_label}\n')
'''

# patients present in both wsi and label
common_patients = list(set(label_not_wsi).intersection(set(label_not_omics)))
print(f'Number of patients present in neither wsi and omics datasets: {len(common_patients)}')

Number of patients present in neither wsi and omics datasets: 0


### events- censorship

how many events/censor patients 

min-max of time event

mean time event



In [56]:
censorship = 'censorship_os'
label_var = 'survival_months_os'

In [57]:
# print info on censorship of label data
print('--- CENSORSHIP ---')
censorship_counts = label_data[censorship].value_counts()
print(censorship_counts)
print('percentage of censored patients: ', censorship_counts[1] / censorship_counts.sum())

--- CENSORSHIP ---
censorship_os
1    409
0     77
Name: count, dtype: int64
percentage of censored patients:  0.8415637860082305


In [58]:
# print info on survival months of label data
print('--- SURVIVAL MONTHS ---')
survival_months_stats = label_data[label_var].describe()
print(survival_months_stats)
print('min: ', label_data[label_var].min())
print('max: ', label_data[label_var].max())
print('average: ', label_data[label_var].mean())

--- SURVIVAL MONTHS ---
count    486.000000
mean      38.656975
std       30.337073
min        0.070000
25%       17.297500
50%       30.270000
75%       54.077500
max      225.330000
Name: survival_months_os, dtype: float64
min:  0.07
max:  225.33
average:  38.656975308641975


In [59]:
print('num of patients with negative survival months: ', len(label_data[label_data[label_var] < 0]))

num of patients with negative survival months:  0


In [60]:
# info on age and sex of patients in label data
print('--- AGE ---')
age_stats = label_data['age'].describe()
print(age_stats)
print('--- SEX ---')
sex_counts = label_data['is_female'].value_counts()
print(sex_counts) 
# percentage of female
print('percentage of female patients: ', sex_counts[1] / sex_counts.sum())

--- AGE ---
count    484.000000
mean      63.807851
std       11.127078
min       31.000000
25%       57.000000
50%       63.000000
75%       71.000000
max       90.000000
Name: age, dtype: float64
--- SEX ---
is_female
1    486
Name: count, dtype: int64
percentage of female patients:  1.0
